# Concepts: how semantic search works

This notebook explains the ideas behind the GIF search project. It needs **no dataset and no index**, only the installed requirements. Run the cells from top to bottom. It takes about 20 minutes.

By the end you will know:
1. What an *embedding* (vector) is.
2. How to measure whether two sentences are similar.
3. How to build a search engine in ~10 lines.
4. Why we use a vector database (Chroma).
5. What this kind of search can and cannot do.

## 1. Turning text into numbers

A computer cannot compare the *meaning* of two sentences. It can compare numbers. An **embedding model** reads a sentence and outputs a list of numbers, called a **vector**. We use `all-MiniLM-L6-v2`, which outputs 384 numbers per sentence.

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

vector = model.encode("a dog being confused")
print("shape:", vector.shape)         # (384,) = 384 numbers
print("first 5 numbers:", vector[:5])

Loading weights: 100%|█████████████████| 103/103 [00:00<00:00, 10726.85it/s]


shape: (384,)
first 5 numbers: [-0.01593952 -0.02276123  0.01635111  0.04333459 -0.0322747 ]


The individual numbers mean nothing on their own. What matters is the **relationship between vectors**: sentences with similar meaning end up with similar vectors.

## 2. Similar meaning = similar vectors

**Cosine similarity** measures how much two vectors point in the same direction. The result is between -1 and 1 (for text, usually between 0 and 1). Higher means more similar.

In [4]:
from sentence_transformers import util

sentences = [
    "a puppy tilts its head",
    "a dog looks confused",
    "a cat plays with a ball",
    "the stock market crashed",
]
vectors = model.encode(sentences)

# Compare every sentence with every other sentence.
scores = util.cos_sim(vectors, vectors)

for i, a in enumerate(sentences):
    for j, b in enumerate(sentences):
        if i < j:
            print(f"{scores[i][j].item():.2f}  {a!r}  vs  {b!r}")

0.51  'a puppy tilts its head'  vs  'a dog looks confused'
0.17  'a puppy tilts its head'  vs  'a cat plays with a ball'
0.06  'a puppy tilts its head'  vs  'the stock market crashed'
0.20  'a dog looks confused'  vs  'a cat plays with a ball'
0.03  'a dog looks confused'  vs  'the stock market crashed'
-0.01  'a cat plays with a ball'  vs  'the stock market crashed'


**Check yourself:** the first two sentences share no words, but the score is high. Why is that useful for GIF search? *(Answer: you can find "a dog looks confused" by typing "puppy tilts head". Keyword search could not.)*

## 3. Cosine similarity by hand

Here is the formula in plain Python with tiny 2D vectors, so you can see there is no magic. Cosine similarity is the dot product divided by the lengths of both vectors.

In [5]:
import numpy as np

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(cosine([1, 0], [1, 0]))   # same direction     -> 1.0
print(cosine([1, 0], [1, 1]))   # 45 degrees apart   -> 0.71
print(cosine([1, 0], [0, 1]))   # 90 degrees apart   -> 0.0

# It gives the same answer as the library, on real 384-number vectors:
print(cosine(vectors[0], vectors[1]), util.cos_sim(vectors[0], vectors[1]).item())

1.0
0.7071067811865475
0.0
0.50715685 0.5071568489074707


## 4. A search engine in ~10 lines

Search = encode the query, compare it with every stored vector, return the best ones. This is called **brute-force** search.

In [6]:
descriptions = [
    "a dog is tilting his head in confusion",
    "a man dressed in red is dancing",
    "a cat tries to catch a mouse on a tablet",
    "a woman laughs at a joke",
    "two dogs are wearing funny hats",
    "a baby panda falls down a hill",
    "a man is glaring at the camera",
    "a kitten is sleeping on a keyboard",
]
stored_vectors = model.encode(descriptions)   # done once, like index.py

def brute_force_search(query, k=3):
    query_vector = model.encode(query)
    scores = util.cos_sim(query_vector, stored_vectors)[0]   # one score per stored sentence
    best = scores.argsort(descending=True)[:k]               # positions of the top-k scores
    return [(descriptions[i], scores[i].item()) for i in best]

for text, score in brute_force_search("a confused puppy"):
    print(f"{score:.2f}  {text}")

0.59  a dog is tilting his head in confusion
0.30  two dogs are wearing funny hats
0.19  a baby panda falls down a hill


In [7]:
# Try your own queries. Which ones work well? Which fail?
for text, score in brute_force_search("cute animals sleeping"):
    print(f"{score:.2f}  {text}")

0.46  a kitten is sleeping on a keyboard
0.41  two dogs are wearing funny hats
0.28  a baby panda falls down a hill


## 5. Why a vector database?

Brute force compares the query with **every** vector. That is fine for 8 sentences, slow for 125,000, and painful for millions. A **vector database** stores the vectors in a smart structure (Chroma uses an index called HNSW) so it can find *approximately* the nearest ones without checking them all. It also stores the extra data (description, GIF url) next to each vector and saves everything to disk.

Below is the same search with Chroma. This is what `index.py` and `search.py` do, at small scale. The database lives in memory only, so nothing is saved to disk.

In [8]:
import chromadb

client = chromadb.EphemeralClient()   # in-memory database (the project uses PersistentClient = saved on disk)
collection = client.get_or_create_collection("demo", metadata={"hnsw:space": "cosine"})

# "Indexing": store vectors, the original text, and extra info.
collection.upsert(
    ids=[str(i) for i in range(len(descriptions))],
    embeddings=stored_vectors.tolist(),
    documents=descriptions,
    metadatas=[{"url": f"https://example.com/{i}.gif"} for i in range(len(descriptions))],
)

# "Searching": ask for the 3 nearest vectors.
result = collection.query(query_embeddings=model.encode("a confused puppy").tolist(), n_results=3)

for text, meta, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
    print(f"score {1 - distance:.2f}  {text}  {meta['url']}")

ModuleNotFoundError: No module named 'chromadb'

Notice `1 - distance`. Chroma returns a **distance** (small = close). We flip it into a **score** (big = similar) so results are easier to read. Compare the scores with the brute-force ones above. They match.

## 6. What this search can and cannot do

The model only reads the **text descriptions**. It never looks at the GIF pixels. So:
- Good: paraphrases and related ideas ("puppy" matches "dog").
- Weak: negation ("a dog that is *not* happy"), exact counts, rare names.
- The result quality depends on the descriptions. If a description is bad, the GIF is hard to find.

In [9]:
for query in ["a dog that is not happy", "three dogs", "a sleepy baby animal"]:
    print("QUERY:", query)
    for text, score in brute_force_search(query, k=2):
        print(f"  {score:.2f}  {text}")

QUERY: a dog that is not happy
  0.53  a dog is tilting his head in confusion
  0.34  two dogs are wearing funny hats
QUERY: three dogs
  0.40  two dogs are wearing funny hats
  0.32  a dog is tilting his head in confusion
QUERY: a sleepy baby animal
  0.40  a dog is tilting his head in confusion
  0.37  a baby panda falls down a hill


## Exercises

1. Add 5 more descriptions to the list in section 4 and re-run the searches.
2. Find two sentences that mean the same thing but score below 0.5. What went wrong?
3. Find two sentences with almost the same words but opposite meaning (for example "I love it" and "I hate it"). What is their score? What does that tell you about the model?
4. Change `k` in `brute_force_search`. What happens to the scores at the bottom of the list?

## Next step

You now know every idea in the project. Open the real code in this order: `data.py`, `index.py`, `search.py`, `app.py`. Then run `notebooks/gif_search.ipynb` to search 5,000 real GIFs.